In [2]:
import pandas as pd
import ast
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

c:\Users\bhaga\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\bhaga\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
df = pd.read_csv(r"C:\Users\bhaga\.cache\kagglehub\datasets\grouplens\movielens-20m-dataset\versions\1\df.csv")
df = df.iloc[:, 1:]

# One row per movie — drop duplicate movieIds, keep first occurrence
movies = df.drop_duplicates('movieId').set_index('movieId')

In [4]:
# Parse genome_tags column from string repr of dict -> actual dict
def parse_tags(val):
    if pd.isna(val):
        return {}
    try:
        return ast.literal_eval(val)
    except Exception:
        return {}    

In [5]:
tag_dicts = movies['genome_tags'].apply(parse_tags)

# Build a tag feature matrix: rows=movies, cols=unique tags, values=relevance scores
tag_df = pd.DataFrame(list(tag_dicts), index=movies.index).fillna(0)

# Cosine similarity between all movies based on genome tag relevance
similarity_matrix = cosine_similarity(tag_df)
similarity_df = pd.DataFrame(similarity_matrix, index=tag_df.index, columns=tag_df.index)

print(f"Tag feature matrix: {tag_df.shape}")
print(f"Similarity matrix: {similarity_df.shape}")

Tag feature matrix: (21940, 1102)
Similarity matrix: (21940, 21940)


In [6]:
np.save("models/similarity_matrix.npy", similarity_matrix)

In [9]:
import pickle

In [10]:
pickle.dump(movies[["title"]], open("models/movies.pkl", "wb"))


In [ ]:
recommend_merged(15)

,movieId,title,final_score,als_score,content_score,num_ratings,vote_average
0,3993,Quills (2000),0.778703,0.796776,0.286405,61,7.135
5,4326,Mississippi Burning (1988),0.762452,0.467755,0.437490,113,7.700
3,678,Some Folks Call It a Sling Blade (1993),0.747719,0.490536,0.426855,94,6.691
10,8783,"Village, The (2004)",0.720900,0.395940,0.448362,83,6.487
14,8957,Saw (2004),0.716552,0.373274,0.423880,157,7.400
36,110,Braveheart (1995),0.705161,0.252253,0.440173,1837,7.900
1,2770,Bowfinger (1999),0.698249,0.517100,0.354419,109,6.187
15,1179,"Grifters, The (1990)",0.697709,0.341956,0.444998,162,6.512
7,532,Serial Mom (1994),0.684599,0.417001,0.395287,83,6.730
4,1876,Deep Impact (1998),0.681874,0.470577,0.358039,97,6.238


In [6]:
def recommend_similar(movie_id, n=10):
    if movie_id not in similarity_df.index:
        raise ValueError(f"movieId {movie_id} not found")
    scores = similarity_df[movie_id].drop(movie_id).sort_values(ascending=False).head(n)
    results = movies.loc[scores.index, 'title'].copy()
    results = pd.DataFrame({'title': results, 'similarity': scores})
    return results

In [7]:
# Example
recommend_similar(260)  # Star Wars: Episode IV

,title,similarity
movieId,,
1196,Star Wars: Episode V - The Empire Strikes Back...,0.699921
1210,Star Wars: Episode VI - Return of the Jedi (1983),0.601839
7988,Space Truckers (1996),0.530986
4198,Battle Beyond the Stars (1980),0.520544
113345,Jupiter Ascending (2015),0.510919
40697,Babylon 5,0.510406
3190,Supernova (2000),0.509990
8633,"Last Starfighter, The (1984)",0.508802
63433,Farscape: The Peacekeeper Wars (2004),0.507964
